# 基本面量化

In [ ]:
# xtquant
from xtquant import xtdata
from xtquant.xttrader import XtQuantTrader
from xtquant import xtconstant
from xtquant.xttrader import XtQuantTrader, XtQuantTraderCallback
from xtquant.xttype import StockAccount

# self defined
#import backtest as bt
#from backtest import Context 

# data science
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns 
# system
import datetime
import time
import logging
import random


***<font color=steel
    size=5
       face=雅黑>
<mark>获取目标股票池</mark>
</font>***

In [ ]:
#xtdata.download_sector_data()
#sectors = xtdata.get_sector_list()
sector_list = ['创业板', '沪深A股', '沪深ETF', '科创板',
               'ETF主题指数', 'ETF债券型', 'ETF商品型', 
               'ETF股票型', 'ETF行业指数', 'ETF货币型', 'ETF跨境型']
sector_list = ['沪深300']
res = {}
for sector in sector_list:
    stock_list = xtdata.get_stock_list_in_sector(sector)
    for stock in stock_list:
        res[stock] = 1

In [ ]:
print(res.keys())

In [ ]:
def get_inst_detail(code):
    return xtdata.get_instrument_detail(code)

for key in res.keys():
    res[key] = get_inst_detail(key)

res

In [ ]:
list(res.keys())

In [ ]:
stock_list = list(res.keys())
xtdata.download_financial_data2(stock_list, table_list=[], start_time='', end_time='', callback=None)
financial_data = xtdata.get_financial_data(stock_list, table_list=[], start_time='', end_time='', report_type='report_time')
financial_data

In [ ]:
period = '1d'
start_date_str = '20250101'
end_date_str = '20251231'
fields=['open', 'close', 'high', 'low', 'volume', 'amount', 'preClose']
xtdata.download_history_data2(stock_list, period, start_date_str, end_date_str)
    
market_data = xtdata.get_market_data_ex(field_list=fields,
                                    stock_list=stock_list,
                                    period=period,
                                    start_time='',
                                    end_time=end_date_str,
                                    count=-1,
                                    dividend_type='none'
            )

In [ ]:
for stock_code in stock_list:
    market_test = market_data[stock_code]
    financial_test = financial_data[stock_code]['PershareIndex']
    market_test['change_time'] = market_test.index
    market_test['change_time'] = market_test['change_time'].apply(lambda x: x[:6])
    financial_test['change_time'] = financial_test['m_timetag'].apply(lambda x: x[:6])
    test = pd.merge(market_test, financial_test, on='change_time', how='left')
    test = test.ffill()
    test['pb_ratio'] = test['close'] / test['s_fa_bps']
    test['pb_ratio'].plot()